In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [3]:
df = pd.read_csv("datasets/studentknndata.csv")
display(df)

,Student,Attendance,PreviousMarks,AssignmentCompletion,QuizScore,Pass
0,Aarav,72,58,65,61,0
1,Sita,91,84,86,88,1
2,Ram,68,52,55,57,0
3,Anisha,94,89,92,91,1
4,Rohan,75,60,62,64,1
5,Priya,88,81,79,85,1
6,Kiran,62,45,48,50,0
7,Nabin,82,69,72,74,1
8,Suman,70,54,58,60,0
9,Alisha,96,93,94,95,1


In [4]:
features = ["Attendance", "PreviousMarks", "AssignmentCompletion", "QuizScore"]

X = df[features]
y = df["Pass"]

print("Features (X):")
display(X.head())

print("Target (y):")
display(y.head())

Features (X):


,Attendance,PreviousMarks,AssignmentCompletion,QuizScore
0,72,58,65,61
1,91,84,86,88
2,68,52,55,57
3,94,89,92,91
4,75,60,62,64


Target (y):


0    0
1    1
2    0
3    1
4    1
Name: Pass, dtype: int64

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 16
Testing samples: 4


## 5. Apply Feature Scaling

KNN uses distance to measure similarity. Since all four features are scores but can still have different distributions, we standardize them using `StandardScaler`.

In [6]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [7]:
knn = KNeighborsClassifier(n_neighbors=3)

knn.fit(X_train_scaled, y_train)

print("KNN model trained successfully.")

KNN model trained successfully.


In [8]:
y_pred = knn.predict(X_test_scaled)

results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

results


,Actual,Predicted
0,0,0
1,0,0
2,1,1
3,1,1


In [9]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"Accuracy : {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall   : {recall:.2f}")
print(f"F1-score : {f1:.2f}")

Accuracy : 1.00
Precision: 1.00
Recall   : 1.00
F1-score : 1.00


## Find the 3 most similar students to a new student

For demonstration, the new student's data is:

- Attendance = **80**
- PreviousMarks = **70**
- AssignmentCompletion = **75**
- QuizScore = **76**

You can change these four values to the values given by your teacher.

In [10]:
new_student = pd.DataFrame(
    [[80, 70, 75, 76]],
    columns=features
)

# Scale the complete existing dataset and the new student
full_scaler = StandardScaler()
X_all_scaled = full_scaler.fit_transform(X)

new_student_scaled = full_scaler.transform(new_student)

# KNN model using all existing students
similarity_model = KNeighborsClassifier(n_neighbors=3)
similarity_model.fit(X_all_scaled, y)

# Find the 3 nearest existing students
distances, indices = similarity_model.kneighbors(new_student_scaled)

nearest_students = df.iloc[indices[0]].copy()
nearest_students["Distance"] = distances[0]

nearest_students[["Student"] + features + ["Pass", "Distance"]]

,Student,Attendance,PreviousMarks,AssignmentCompletion,QuizScore,Pass,Distance
7,Nabin,82,69,72,74,1,0.295606
12,Roshan,85,72,75,78,1,0.474404
15,Rita,77,66,68,71,1,0.658529


In [11]:
new_prediction = similarity_model.predict(new_student_scaled)[0]

if new_prediction == 1:
    print("Prediction: The new student is likely to PASS.")
else:
    print("Prediction: The new student is likely to FAIL.")

Prediction: The new student is likely to PASS.


## Conclusion

KNN predicts the new student's result by comparing the student with nearby existing students in the scaled feature space. The three nearest students are the most similar based on attendance, previous marks, assignment completion, and quiz score.
